In [8]:
import pandas as pd
import re
from docx import Document
import matplotlib.pyplot as plt


In [4]:
# 1. Cargamos el alineamiento Clustal (lo ideal es leerlo del archivo generado por MAFFT)
def load_alignment(file_path):
    sequences = {"Humano_Alp": "", "7UUY_Raton": "", "7UUZ_Raton": "", "7UV0_Raton": ""}
    with open(file_path, "r") as f:
        for line in f:
            for key in sequences.keys():
                if line.startswith(key):
                    parts = line.split()
                    if len(parts) > 1:
                        sequences[key] += parts[1]
    return sequences

# 2. Diccionario de conversión de aminoácidos
d3_to_1 = {
    'Gly':'G','Arg':'R','Leu':'L','Val':'V','Ala':'A','Asp':'D','Asn':'N','Glu':'E','Gln':'Q','Met':'M',
    'Ile':'I','Thr':'T','Ser':'S','Phe':'F','Tyr':'Y','Cys':'C','Trp':'W','His':'H','Pro':'P','Lys':'K'
}

def parse_docx_variants(docx_path):
    doc = Document(docx_path)
    # Extraemos texto de párrafos y tablas
    full_text = "\n".join([p.text for p in doc.paragraphs])
    for table in doc.tables:
        for row in table.rows:
            full_text += "\n" + "\n".join([cell.text for cell in row.cells])
    
    # Buscamos variantes tipo p.Gly18Arg
    return re.findall(r"p\.([A-Z][a-z]{2})(\d+)([A-Z][a-z]{2})", full_text)

def run_mapping(docx_file, alignment_file):
    aln = load_alignment(alignment_file)
    variants = parse_docx_variants(docx_file)
    
    # Mapeo Humano: Residuo -> Columna de alineamiento
    hum_to_aln = {}
    h_count = 0
    for col, char in enumerate(aln["Humano_Alp"]):
        if char != "-":
            h_count += 1
            hum_to_aln[h_count] = col

    for pdb_key in ["7UUY_Raton", "7UUZ_Raton", "7UV0_Raton"]:
        foldx_list = []
        log_data = []
        
        # Mapeo PDB Ratón: Columna -> Residuo real (saltando gaps)
        # Esto es vital por el tag inicial de las estructuras
        aln_to_pdb = {}
        r_count = 0
        for col, char in enumerate(aln[pdb_key]):
            if char != "-":
                r_count += 1
                aln_to_pdb[col] = r_count

        for ref3, pos, mut3 in variants:
            h_pos = int(pos)
            if h_pos not in hum_to_aln:
                log_data.append(f"Variante p.{ref3}{pos}{mut3}: Fuera de rango del alineamiento.")
                continue
            
            col_idx = hum_to_aln[h_pos]
            
            if col_idx not in aln_to_pdb:
                log_data.append(f"Variante p.{ref3}{pos}{mut3}: Cae en un GAP en {pdb_key}.")
                continue
            
            pdb_pos = aln_to_pdb[col_idx]
            wt_aa = aln[pdb_key][col_idx] # El AA que está en el PDB
            
            try:
                mut_1 = d3_to_1[mut3]
                # Formato FoldX: WT + Cadena + Posición + Mutante;
                # Asumimos cadena A. Si es otra, cambiar 'A' por la letra correspondiente.
                foldx_list.append(f"{wt_aa}A{pdb_pos}{mut_1};")
            except KeyError:
                log_data.append(f"Variante p.{ref3}{pos}{mut3}: Error en código de aminoácido.")

        # Guardar lista para FoldX
        with open(f"individual_list_{pdb_key}.txt", "w") as f:
            f.write("\n".join(foldx_list))
        
        # Guardar Log de control
        with open(f"log_mapeo_{pdb_key}.txt", "w") as f:
            f.write("\n".join(log_data))

    print("--- Proceso Finalizado ---")
    print(f"Se procesaron {len(variants)} variantes del DOCX.")
    print("Revisá los archivos 'individual_list_*.txt' para FoldX.")


In [6]:

# Para ejecutar:
run_mapping("SAVYSUPPLEMENTALTABLE2.docx", "alineamiento.txt")

--- Proceso Finalizado ---
Se procesaron 39 variantes del DOCX.
Revisá los archivos 'individual_list_*.txt' para FoldX.


In [9]:
# Datos de ejemplo basados en tu output de FoldX y la tabla humana
# (Esto simula lo que el script compararía internamente)
comparativa = {
    "ID_Variante": ["p.Gly18Arg", "p.Gly93Arg", "p.Asn157Thr", "p.Asp191Gly"],
    "AA_Humano": ["G", "G", "N", "D"],
    "AA_Raton_PDB": ["G", "A", "N", "D"], # Notar que en pos 93 el ratón tiene Ala
    "Pos_Humana": [18, 93, 157, 191],
    "Pos_PDB_Raton": [34, 109, 173, 207]
}

df_comp = pd.DataFrame(comparativa)

def comprobar_consistencia(df):
    # Verificamos mutaciones donde el ratón difiere del humano nativamente
    differences = df[df["AA_Humano"] != df["AA_Raton_PDB"]]
    print(f"✅ Se encontraron {len(differences)} posiciones donde el Ratón difiere del Humano nativamente.")
    return differences

# Visualización de la distribución de variantes
def plot_variantes(df):
    plt.figure(figsize=(10, 4))
    plt.scatter(df["Pos_Humana"], [1]*len(df), alpha=0.5, c='blue', label='Variantes Mapeadas')
    plt.title("Distribución de Variantes a lo largo de la secuencia Humana")
    plt.xlabel("Posición Residuo Humano")
    plt.yticks([])
    plt.grid(axis='x', linestyle='--', alpha=0.7)
    plt.show()

diffs = comprobar_consistencia(df_comp)
print(diffs)
# plot_variantes(df_comp) # Descomentar en local

✅ Se encontraron 1 posiciones donde el Ratón difiere del Humano nativamente.
  ID_Variante AA_Humano AA_Raton_PDB  Pos_Humana  Pos_PDB_Raton
1  p.Gly93Arg         G            A          93            109
